# Peak search of a map of Laue patterns

Find the Laue spots (peaks) of every image of a map, using multiprocessing (1 image per cpu) locally or as a job on the ESRF cluster (SLURM).

Results: `prefix####.dat` (peak list: X, Y, intensity, shape ...) and `prefix####.cor` (same + scattering angles 2θ, χ, from the detector calibration `.det` file), 1 file of each per image

**This notebook is part of the LaueTools package** (https://github.com/BM32ESRF/lauetools)

Author: J.-S. Micha, Last revision: Oct 2026. Tested with python 3.12 (jupyterlab, jupyterhub, VS Code)

| notebook | input | output |
|---|---|---|
| `peaksearch/peaksearch_map.ipynb` | images | `.dat`, `.cor` files (peak lists) |
| `indexation/1_index_refine.ipynb` | `.cor` files | `.fit` files (indexed spots, UB matrix, strain per grain) |
| `indexation/2_postprocess_maps.ipynb`, `3_segmentation_grains.ipynb` | `.fit` files | maps, grains |

**All settings of an experiment are in a YAML configuration file**: copy `config_template.yaml` to `configs/my_experiment.yaml`, edit it and set `CONFIG` in the first code cell. At the end, this notebook writes the configuration file of the indexation (same folders, detector, map).

Batch mode (optional, no browser): `papermill peaksearch_map.ipynb run_myexp.ipynb -p CONFIG configs/my_experiment.yaml`

# SETTINGS

In [ ]:
CONFIG = 'configs/ma6034_Zn.yaml'   # configuration file of your experiment

NB_CPUS = None             # None: nb_cpus of the configuration file
RUN_BATCH = True           # False: only tests on 1 image
SAVE_RESULTS = True        # save peaksearch_<stamp>.pickle (all peak lists of the batch)
OVERWRITE_RESULTS = False  # True: overwrite an existing peaksearch_<stamp>.pickle
EXECUTION = 'local'        # 'local': batch in this jupyter session; 'slurm': job(s) on the ESRF cluster (section BATCH on SLURM)

# IMPORTS

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"   # before numpy import: 1 thread per process (images are dispatched on cpus)
import time
from pathlib import Path

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

try:
    import ipympl  # interactive plots (zoom, hover) in JupyterLab / VS Code
    %matplotlib widget
except ImportError:
    %matplotlib inline

import LaueTools.peaksearch_batch as PB
import LaueTools.generaltools as GT
import LaueTools.dict_LaueTools as DictLT

print('LaueTools from', Path(PB.__file__).parent)
print(f'nb of cpus available: {PB.available_cpus()}')

# LOAD CONFIGURATION

After editing the configuration file, run this cell again.

In [ ]:
cfg = PB.load_config(CONFIG)
PB.print_config(cfg)
config_ok = PB.check_config(cfg)

scan = cfg['scan']
mapdims, invmapdims = scan['mapdims'], scan['invmapdims']   # (fast, slow), (slow, fast)
dat_folder = cfg['output']['dat_folder']

## [OPTION] scans of the experiment from the BLISS hdf5 file

to find the `dmesh` command of the map and set `mapdims` = [nbsteps motor1 + 1, nbsteps motor2 + 1] in the configuration file

In [ ]:
if scan['hdf5_logfile'] is not None and Path(scan['hdf5_logfile']).exists():
    import pandas as pd
    import LaueTools.logfile_reader as iohdf5
    listscans, listcts = iohdf5.get_scans_cts(str(scan['hdf5_logfile']))
    dfallscans = pd.DataFrame(listscans, columns=['start_time', 'end_time', 'sample_dataset_scanindex', 'fullcommand',
                                                  'scanindex', 'scantype', 'motors', 'localhdf5file', 'imagefolder'])
    print(f'{len(listscans)} scans and {len(listcts)} saved count commands found')
    display(dfallscans[dfallscans['scantype'].astype(str).str.contains('mesh')][['start_time', 'sample_dataset_scanindex', 'fullcommand']])
else:
    print('no hdf5 log file (scan: hdf5_logfile in the configuration file)')

# TEST on 1 image

Tune the peak search parameters (section `peaksearch` of the configuration file) on a few images before the batch:

- `IntensityThreshold`: minimum amplitude of a peak above the background. Lower: more (weak) peaks, but slower and more noise
- `boxsize`, `PeakSizeRange`, `FitPixelDev`: fitted region and accepted peak sizes / shifts (pixel)
- `background: auto` removes a background computed from the image itself (minimum filter, `formulaexpression: A-1.1*B`)
- `exclusion_boxes`: rectangles (e.g. detector gaps, beamstop shadow) where peaks are removed; `npixels`: remove peaks close to the gaps between detector modules (sCMOS, EIGER_4MCdTe)
- parameters of a `.psp` file saved by the LaueTools GUI: `cfg = PB.import_pspfile(cfg, 'my.psp')` writes them in the configuration file

Parameters can also be changed here for tests (then save the good ones with `PB.save_config`, cell after the plots):
`params_test.peaksearch['IntensityThreshold'] = 300`, `params_test.exclusion_boxes = [[1009, 1000, 7, 1100]]`, ...

In [ ]:
test_image_index = cfg['run']['test_image_index']
imagefile = PB.image_path(cfg, test_image_index)

params_test = PB.params_from_config(cfg, writefiles=False, verboselevel=1)   # writefiles=True: write .dat/.cor of the test
# params_test.peaksearch['IntensityThreshold'] = 300

t0 = time.time()
res = PB.peaksearch_image(imagefile, params_test)
PB.print_result(res)
print(f'elapsed time {time.time() - t0:.2f} s')
peaklist = res[2]

## PLOT image and found peaks

If some spots are missed, decrease `IntensityThreshold`; if there are peaks in the noise, increase it.

In [ ]:
imagedata = PB.read_image(imagefile, scan['CCDLabel'])
ax = PB.plot_peaks(imagedata, peaklist, params_test.exclusion_boxes, title=f'{imagefile.parent}\n{imagefile.name}')
# ax = PB.plot_peaks(imagedata, peaklist, params_test.exclusion_boxes, vmin=1000, vmax=1500)   # other color scale
plt.show()

In [ ]:
# most intense peaks and their fitted ellipse (fwhm)
fig = PB.plot_vignettes(imagedata, peaklist, nbmax=100)
plt.show()

In [ ]:
# [OPTION] the test is good: save the tested parameters in a NEW configuration file (comments kept, no manual YAML editing)
# Only the parameters of params_test that differ from cfg are written (printed below). The batch (params_from_config(cfg)) then uses them.
SAVE_TESTED_CONFIG = False
NEW_CONFIG = 'configs/my_experiment_v2.yaml'

if SAVE_TESTED_CONFIG:
    cfg = PB.save_config(cfg, NEW_CONFIG, params=params_test)   # other changes, e.g.: test_image_index=1300
    CONFIG = str(cfg['configfile'])
    dat_folder = cfg['output']['dat_folder']
    PB.print_config(cfg)

## [OPTION] test on a few successive images

In [ ]:
TEST_SEVERAL = True
if TEST_SEVERAL:
    params_test.verboselevel = 0
    for imageindex in range(test_image_index, test_image_index + 5):
        PB.print_result(PB.peaksearch_image(PB.image_path(cfg, imageindex), params_test))

# BATCH: PEAK SEARCH of all images (multiprocessing)

- set of images: section `selection` of the configuration file (all images, `image_range` or `roi`)
- `run: overwrite: false` analyses only images without `.dat` file (e.g. to resume an interrupted batch)
- `.cor` files are written if `scan: detfile` (calibration) is given: they are the input of the indexation
- large maps (OPTION): `output: nbfiles_per_folder: 10000` writes the files in subfolders `images_0_9999/`, `images_10000_19999/` ... of `dat_folder` (the indexation configuration written at the end reads them there)
- `EXECUTION = 'slurm'` (SETTINGS): the batch runs as a job on the ESRF cluster, see section BATCH on SLURM (the cell below then only prepares `listfiles` and `params`)

In [ ]:
params = PB.params_from_config(cfg)   # parameters of the configuration file (changes of params_test are not used)

listfiles = PB.select_images(cfg)
print(f'{len(listfiles)} images to analyse, from {Path(listfiles[0]).name} to {Path(listfiles[-1]).name}')
if mapdims is not None:
    print(f'map {mapdims[0]} x {mapdims[1]} = {mapdims[0] * mapdims[1]} images')
print(f'peak lists -> {params.dat_folder}' + ('' if params.calibration else '  (no .cor file: no calibration detfile)'))

## [OPTION] BATCH on LOCAL machine

With `EXECUTION = 'local'` in SETTINGS, the batch runs on the current machine (jupyter session)

In [ ]:
nb_cpus = NB_CPUS or cfg['run']['nb_cpus'] or PB.available_cpus()
print(f'{nb_cpus} processes, {PB.available_cpus()} cpus available')

allresults = None
if RUN_BATCH and config_ok and EXECUTION == 'local':
    allresults = PB.run_multiprocessing(listfiles, params, nb_cpus=nb_cpus)
    GT.printgreen('Done!')

## [OPTION] BATCH on SLURM (ESRF cluster)

With `EXECUTION = 'slurm'` in SETTINGS, the batch runs as a job on a node of the cluster (1 node, 1 process per cpu), not in this jupyter session. The job uses **exactly** `params` and `listfiles` of the cell above, including the changes made in the notebook.

1. **prepare**: writes a job folder `slurm_jobs/<job name>_<date>/` next to the folder of peak lists (`params.dat_folder`) with the parameters (`job_params.pickle`, readable copy `job_params.txt`) and the slurm script `job.slurm`. Logs (`logs/`) and results (`allresults_part###.pickle`) of the job are written there too
2. **submit**: `SUBMIT = True` (sbatch from this notebook, in a jupyter-slurm session), or copy the printed `sbatch ...` command in a terminal of jupyter-slurm
3. **monitor**: state of the job, progress and last lines of the log (run the cell again)
4. **load results**, then SUMMARY and SAVE below, as for a local batch

| `SLURM_MACHINE` | partition | cpus / node |
|---|---|---|
| `magnifix` | magnifix | 192 (priority) |
| `magnifix2`, `magnifix3` | magnifix | 2 or 3 nodes of 192 cpus (priority): job array of at least 2 or 3 jobs |
| `hpc6`, `hpc7`, `hpc8` | nice (+ constraint) | 96 |
| `nice` | nice | 96 |
| `auto` | magnifix if more than 96 cpus are asked and a magnifix node has enough idle cpus, otherwise the hpc6/7/8 node with most idle cpus (96 cpus) | |

(the same machines as the indexation: `PB.SLURM_MACHINES` = `IB.SLURM_MACHINES`, check with `sinfo -N -o "%N %P %c %m %f %T"`)

- the peak search of an image takes ~1 s x cpu: a local batch is often enough for maps of a few thousand images. SLURM is useful for large maps
- time limit: estimate it with a short local benchmark (below). If the job is stopped by the time limit, prepare and submit again with `params.overwrite = False`: only images without `.dat` file are analysed
- large maps: `NCHUNKS` > 1 splits the images into `NCHUNKS` jobs (slurm job array), which may run at the same time on several nodes

In [ ]:
# [OPTION] time limit estimate from a short local batch (no file written)
BENCHMARK = False
SLURM_CPUS = 96        # cpus per job (1 node): more than 96 -> magnifix
NCHUNKS = 1            # nb of jobs (job array), each one on 1/NCHUNKS of listfiles

if BENCHMARK:
    sec_cpu_per_image = PB.benchmark(listfiles, params, nb_images=2 * PB.available_cpus())
    print('suggested time limit:', PB.walltime_estimate(len(listfiles) / NCHUNKS, SLURM_CPUS, sec_cpu_per_image))

In [ ]:
SLURM_MACHINE = 'auto'       # 'auto', 'magnifix', 'magnifix2', 'magnifix3', 'hpc6', 'hpc7', 'hpc8', 'nice'
SLURM_TIME = '00:30:00'      # time limit hh:mm:ss
SLURM_MEM_PER_CPU = '2000M'  # memory per cpu
SUBMIT = True                # True: sbatch now (jupyter-slurm session). False: only prepare and print the sbatch command

job = None
if EXECUTION == 'slurm' and config_ok:
    if SLURM_MACHINE == 'auto':
        machine, ncpus = PB.choose_slurm_machine(SLURM_CPUS)
    else:
        machine, ncpus = SLURM_MACHINE, SLURM_CPUS
    job = PB.prepare_slurm_job(cfg, params, listfiles, machine=machine, nb_cpus=ncpus, time=SLURM_TIME,
                               mem_per_cpu=SLURM_MEM_PER_CPU, nchunks=NCHUNKS)
    nb_cpus = job['nb_cpus']
    if SUBMIT:
        PB.submit_slurm_job(job)

In [ ]:
# job state, progress and last lines of the log(s): run again to update
# after a restart of the kernel: job = '<job folder>'
if EXECUTION == 'slurm' and job is not None:
    PB.slurm_job_status(job)

In [ ]:
# when the job(s) are finished: results for SUMMARY and SAVE below
if EXECUTION == 'slurm' and job is not None:
    allresults = PB.load_slurm_results(job)

## SUMMARY of the batch

nb of peaks per image and its gradient (large where the Laue pattern changes, e.g. at grain boundaries). Hover over the map to read the image index.

In [ ]:
if allresults is None:
    raise RuntimeError('no batch results: set RUN_BATCH = True (and check the configuration)')
summary = PB.summarize_results(allresults, mapdims)

fig = PB.plot_nbpeaks_map(summary, cfg)
plt.show()

In [ ]:
# image with the largest nb of peaks
treated = summary['nbpeaks'] >= 0
imax = summary['imageindices'][treated][np.argmax(summary['nbpeaks'][treated])]
print(f"max nb of peaks: {summary['nbpeaks'][treated].max()} in image {imax}"
      + (f' (fast, slow) = ({imax % mapdims[0]}, {imax // mapdims[0]})' if mapdims else ''))

## [OPTION] peaks of one image of the batch

`image_index` from the map above (hover). With ipywidgets, a slider browses the images.

In [ ]:
def show_image_peaks(image_index=int(imax)):
    peaks = PB.peaks_of_image(allresults, image_index)
    fig, ax = plt.subplots(figsize=(5, 5))
    if peaks is not None:
        ax.scatter(peaks[:, 0], peaks[:, 1], s=np.clip(peaks[:, 2] / np.amax(peaks[:, 2]) * 100, 5, 100), alpha=0.6)
    ax.set_xlim(0, imagedata.shape[1])
    ax.set_ylim(imagedata.shape[0], 0)
    ax.set_aspect('equal')
    ax.set_title(f"image {image_index}: {0 if peaks is None else len(peaks)} peaks (size ~ intensity)", fontsize=9)
    plt.show()

try:
    from ipywidgets import interact
    interact(show_image_peaks, image_index=(int(summary['imageindices'].min()), int(summary['imageindices'].max())))
except ImportError:
    show_image_peaks()

## SAVE results

`peaksearch_<stamp>.pickle` in `output: results_folder` (default: dat_folder): all peak lists (`allresults`: list of (image index, nb of peaks, peak list array)), parameters and summary. Reload it with `dictresults = PB.load_results(path)`

In [ ]:
if SAVE_RESULTS and allresults is not None:
    PB.save_results(cfg, params, allresults, summary, overwrite=OVERWRITE_RESULTS, nb_cpus=nb_cpus)

# NEXT STEP: configuration file of the indexation

writes the configuration of `indexation/1_index_refine.ipynb` with the folders, detector and map of this peak search (`cor_folder` = `dat_folder`). Give at least the material and the folder of `.fit` files; other indexation parameters can be given here (e.g. `nbGrainstoFind=2`, `e_max=22`) or edited later in the file.

In [ ]:
WRITE_INDEXING_CONFIG = False
INDEXING_CONFIG = '../indexation/configs/my_experiment.yaml'
KEY_MATERIAL = 'Zn'
FIT_FOLDER = Path(dat_folder).parent / 'fitfiles'

if WRITE_INDEXING_CONFIG:
    icfg = PB.indexing_config(cfg, INDEXING_CONFIG, key_material=KEY_MATERIAL, fit_folder=FIT_FOLDER)

**Next step:** `../indexation/1_index_refine.ipynb` with `CONFIG = INDEXING_CONFIG`